In [1]:
!pip install scikit-learn
!pip install transformers==4.38.2 datasets==2.16.1 tokenizers==0.15.2 accelerate==0.27.2 lime evaluate openpyxl
!pip uninstall -y numpy
!pip install numpy==1.26.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 359.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 308.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 278.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 182.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 192.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 292.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 123.8 MB

In [2]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix, precision_recall_curve, auc
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import AutoModelForSequenceClassification
from transformers import RobertaPreTrainedModel,  RobertaModel,  RobertaConfig           # ← changed
from lime.lime_text import LimeTextExplainer
import torch, torch.nn as nn
from datasets import Dataset
import evaluate, gc, time, numpy as np, pandas as pd, datetime, pickle, csv, re, platform, os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from transformers import  AutoModelForSequenceClassification, TrainerCallback, PretrainedConfig
import matplotlib.pyplot as plt
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def r2(x):
    if x == "" or x is None:return ""
    return round(float(x), 2)

def parse_vector_column(vec):
    if pd.isna(vec):return []
    if isinstance(vec, str):
        vec = vec.strip()
        if not vec:return []
        return [float(x) for x in vec.split()]
    return [float(x) for x in vec]

def tokenize_and_align_rationale(examples):
    tokenized = tokenizer( examples[xcolumn], truncation=True,
        padding="max_length", max_length=MAX_LENGTH, return_attention_mask=True )
    new_rationale_targets = []
    for i in range(len(examples[xcolumn])):
        rationale = examples["rationale"][i]
        word_ids = tokenized.word_ids(batch_index=i)
        rationale_vec = []
        for wid in word_ids:
            if wid is None or wid >= len(rationale):
                rationale_vec.append(-100)
            else:
                rationale_vec.append(float(rationale[wid]))
        new_rationale_targets.append(rationale_vec)
    tokenized["rationale_targets"] = new_rationale_targets
    return tokenized

In [3]:
MAX_LENGTH = 128
modelname = "xlmt"
num_labels = 5
modelpath = "./models/"
N_EPOCHS = 5
lamda = 0.1

file_path = "/workspace/Data/UOLDRationales-24k-V2.xlsx"
result_path =  "/workspace/Results/T2-M5-Result.csv"
LOCAL_OUT = f"/workspace/lc_models/{modelname}" 
save_dir = "/workspace/ft_models/t2-m5-xlmt-rat"
MODEL_NAME = "/workspace/models/twitter-xlm-roberta-base"
SAVE_result = "/workspace/Results/t2-m5-results_lime.csv"

df = pd.read_excel(file_path, engine='openpyxl')
df.reset_index( drop=True, inplace=True )
df.Tweet = df.Tweet.astype( 'str' )
print(df.columns, df.shape)
xcolumn = "Tweet"
ycolumn = "Tag"
rationale_column = "Vector"

print(df[ycolumn].value_counts().sort_index())
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
df = df[[xcolumn, ycolumn, rationale_column]].dropna()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gc.collect()
torch.cuda.empty_cache()

df['rationale'] = df[rationale_column].apply(parse_vector_column)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[ycolumn], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df[ycolumn],random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

columns_to_remove = [col for col in train_dataset.column_names if col not in [xcolumn, "rationale", "Tag"]]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "Offset mapping requires a fast tokenizer"

special_token_ids = set(tokenizer.all_special_ids)
# APPLY TOKENIZATION + RATIONALE ALIGNMENT
train_dataset = train_dataset.map(tokenize_and_align_rationale, batched=True, remove_columns=columns_to_remove)
val_dataset   = val_dataset.map(tokenize_and_align_rationale, batched=True, remove_columns=columns_to_remove)
test_dataset  = test_dataset.map(tokenize_and_align_rationale, batched=True, remove_columns=columns_to_remove)

# Rename label column
train_dataset = train_dataset.rename_column(ycolumn, "labels")
val_dataset   = val_dataset.rename_column(ycolumn, "labels")
test_dataset  = test_dataset.rename_column(ycolumn, "labels")

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "rationale_targets"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "rationale_targets"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "rationale_targets"])


Index(['Unnamed: 0.1', 'Unnamed: 0', 'TweetId', 'Tweet', 'Vector', 'Words',
       'TweetText', 'TweetSpaceInserted', 'Tag'],
      dtype='str') (24063, 9)
Tag
0    15404
1     4823
2     1168
3     2105
4      563
Name: count, dtype: int64
2026-03-08 05:01:34


Map:   0%|          | 0/17325 [00:00<?, ? examples/s]

Map:   0%|          | 0/1925 [00:00<?, ? examples/s]

Map:   0%|          | 0/4813 [00:00<?, ? examples/s]

In [4]:
from transformers import XLMRobertaPreTrainedModel, XLMRobertaConfig,XLMRobertaModel
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    acc = evaluate.load("accuracy").compute(predictions=preds, references=labels)["accuracy"]
    return {
        "accuracy": acc,
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
}

class XlmtMulticlassExplain(XLMRobertaPreTrainedModel):
    config_class = XLMRobertaConfig
    base_model_prefix = "roberta"

    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.num_labels = config.num_labels
        self.roberta = XLMRobertaModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.classifier.weight.data.normal_(mean=0.0, std=0.02)
        self.classifier.bias.data.zero_()
        self.register_buffer("pos_weight", torch.tensor(config.num_labels, dtype=torch.float32))
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, rationale_targets=None, output_attentions=None, **kwargs):
        if output_attentions is None:
            output_attentions = self.config.output_attentions or (rationale_targets is not None)
        outputs = self.roberta(input_ids=input_ids,attention_mask=attention_mask,
            output_attentions=output_attentions,)
        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output) 
        logits = torch.clamp(logits, -20.0, 20.0)
        loss = None
        if labels is not None:
            cls_loss = nn.CrossEntropyLoss(weight=self.pos_weight)
            cls_loss = cls_loss(logits, labels.long())
            attn_loss = torch.tensor(0.0, device=logits.device)
            if rationale_targets is not None and output_attentions:
                last_attentions = outputs.attentions[-1]
                cls_attentions = last_attentions.mean(dim=1)[:, 0, :]       
                cls_attentions = cls_attentions * attention_mask.float()
                attn_sum = cls_attentions.sum(dim=-1, keepdim=True).clamp(min=1e-6)
                cls_attentions = cls_attentions / attn_sum
                cls_attentions = torch.nan_to_num(cls_attentions, nan=0.0)

                tau = 1.0
                mask = (rationale_targets != -100) & (attention_mask == 1)
                num_tokens = mask.sum(dim=-1, keepdim=True).clamp(min=1)

                is_positive = (labels != 0).unsqueeze(-1)  
                uniform = (1.0 / num_tokens) * mask.float()
                rationale_soft = torch.softmax(rationale_targets.float() / tau, dim=-1) * is_positive.float()
                softened_targets = torch.where(is_positive, rationale_soft, uniform)

                attn_loss = -(softened_targets * torch.log(cls_attentions + 1e-10)).sum(dim=-1)
                attn_loss = (attn_loss * mask.sum(dim=-1) / num_tokens.squeeze(-1)).mean()

            loss = cls_loss + lamda * attn_loss
        attentions = outputs.attentions if output_attentions else None
        return SequenceClassifierOutput( loss=loss, logits=logits,hidden_states=outputs.hidden_states,
            attentions=attentions, )

In [5]:
from sklearn.utils.class_weight import compute_class_weight
train_labels_array = np.array(train_dataset["labels"])
class_weights = compute_class_weight(
    class_weight='balanced',classes=np.unique(train_labels_array),
    y=train_labels_array )
pos_weight_cpu = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", pos_weight_cpu)

gc.collect()
torch.cuda.empty_cache()

config = XLMRobertaConfig.from_pretrained(MODEL_NAME)
config.num_labels = 5
config.output_attentions = False 
config.use_cache = False
model = XlmtMulticlassExplain.from_pretrained(MODEL_NAME, config=config, use_safetensors=True, ignore_mismatched_sizes=True)
model.config.output_attentions = False
model.config.use_cache = False
model.register_buffer("pos_weight", pos_weight_cpu)
model.to(device)

gc.collect()
torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir=LOCAL_OUT,
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    evaluation_strategy="epoch", logging_steps=100,
    save_strategy="epoch", save_steps=100,
    save_total_limit=2, 
    fp16=True,
    fp16_full_eval=True,
    bf16=False,
    bf16_full_eval=False,
    load_best_model_at_end=True,
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
    weight_decay=0.01,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    metric_for_best_model="f1_macro",)

trainer = Trainer(model=model, args=training_args,train_dataset=train_dataset,
                  eval_dataset=val_dataset,compute_metrics=compute_metrics,)
#trainer.train()
trainer.train(resume_from_checkpoint=True)

trainer.save_model(save_dir)        
tokenizer.save_pretrained(save_dir) 
print(f"✅ Model saved to {save_dir}")
gc.collect()
torch.cuda.empty_cache()

Class weights: tensor([0.3124, 0.9980, 4.1201, 2.2871, 8.5345])


Some weights of XlmtMulticlassExplain were not initialized from the model checkpoint at /workspace/models/twitter-xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pos_weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:2720: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a futur

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,F1 Weighted
3,0.881300,1.066813,0.816623,0.649147,0.816623,0.816105
4,0.693000,1.202031,0.815584,0.649756,0.815584,0.816436


✅ Model saved to /workspace/ft_models/t2-m5-xlmt-rat


In [ ]:
Epoch 	Training Loss 	Validation Loss 	Accuracy 	F1 Macro 	F1 Micro 	F1 Weighted
0 	1.094500 	0.969388 	0.784416 	0.546333 	0.784416 	0.767572
1 	0.996300 	0.934818 	0.800519 	0.602807 	0.800519 	0.800958
2 	0.914200 	0.956160 	0.822338 	0.657622 	0.822338 	0.820119
4 	0.866700 	1.085329 	0.811429 	0.633403 	0.811429 	0.812400

In [5]:
config = RobertaConfig.from_pretrained(save_dir)
config.output_attentions = False
config.use_cache = False
model = XlmtMulticlassExplain.from_pretrained(save_dir, config=config,  ignore_mismatched_sizes=True, use_safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(save_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

gc.collect()
torch.cuda.empty_cache()

You are using a model of type xlm-roberta to instantiate a model of type roberta. This is not supported for all configurations of models and can yield errors.
Some weights of XlmtMulticlassExplain were not initialized from the model checkpoint at /workspace/ft_models/t2-m5-xlmt-rat and are newly initialized because the shapes did not match:
- pos_weight: found shape torch.Size([5]) in the checkpoint and torch.Size([]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, pin_memory=True)
all_logits = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(outputs.logits.cpu().numpy())

preds = np.argmax(np.concatenate(all_logits, axis=0), axis=-1)
labels = np.array(test_dataset["labels"])
report = classification_report(labels, preds, target_names=['NON 0', 'CYB 1', 'HAT 2', 'PRO 3', 'OTH 4'], output_dict=True, zero_division=0)
print("Classification Report", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(report)

gc.collect()
torch.cuda.empty_cache()

Classification Report 2026-03-06 16:45:34
{'NON 0': {'precision': 0.8964968152866242, 'recall': 0.913664394677053, 'f1-score': 0.904999196270696, 'support': 3081.0}, 'CYB 1': {'precision': 0.743298969072165, 'recall': 0.7471502590673575, 'f1-score': 0.7452196382428941, 'support': 965.0}, 'HAT 2': {'precision': 0.5739130434782609, 'recall': 0.5641025641025641, 'f1-score': 0.5689655172413793, 'support': 234.0}, 'PRO 3': {'precision': 0.64, 'recall': 0.5320665083135392, 'f1-score': 0.5810635538261998, 'support': 421.0}, 'OTH 4': {'precision': 0.5365853658536586, 'recall': 0.5892857142857143, 'f1-score': 0.5617021276595745, 'support': 112.0}, 'accuracy': 0.8223561188447953, 'macro avg': {'precision': 0.6780588387381418, 'recall': 0.6692538880892457, 'f1-score': 0.6723900066481487, 'support': 4813.0}, 'weighted avg': {'precision': 0.8192859767717122, 'recall': 0.8223561188447953, 'f1-score': 0.8203024724927472, 'support': 4813.0}}


In [8]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
modelname = "T2-M5-XLMT-rat"
with open(result_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["#################################################"])
    writer.writerow([modelname, timestamp])
    writer.writerow(["Accuracy", r2(report.get("accuracy")), ])
    writer.writerow(["Class", "Precision", "Recall", "F1-Score", "Support"])
    for class_name, metrics in report.items():
        if isinstance(metrics, dict):
            writer.writerow([class_name,r2(metrics.get("precision")),r2(metrics.get("recall")),
                r2(metrics.get("f1-score")), r2(metrics.get("support")),])

In [7]:


def auprc(scores, y_true):
    from sklearn.metrics import average_precision_score
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if len(y_true) == 0 or y_true.sum() == 0 or y_true.sum() == len(y_true):return 0.0
    return average_precision_score(y_true, scores)

def token_f1(y_pred, y_true):
    y_pred = np.asarray(y_pred)
    y_true = np.asarray(y_true)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return f1

def iou(y_pred, y_true):
    pred_pos = np.where(y_pred == 1)[0]
    gold_pos = np.where(y_true == 1)[0]
    if len(pred_pos) == 0 or len(gold_pos) == 0:return 0.0
    intersection = len(set(pred_pos) & set(gold_pos))
    union = len(set(pred_pos) | set(gold_pos))
    return intersection / union if union > 0 else 0.0

def subword_to_word_importance(token_scores, word_ids):
    if len(token_scores) == 0: return np.array([])
    word_importance = []
    current_word_id = None
    current = 0 
    for i, wid in enumerate(word_ids):
        if wid is None:  continue
        if wid != current_word_id:
            if current_word_id is not None: word_importance.append(current)
            current_word_id = wid
            current = token_scores[i]
        else:
            current = current + token_scores[i] 
    if current_word_id is not None: word_importance.append(current)
    return np.array(word_importance)

def binarize_topk_nonzero(scores, k):
    scores = np.asarray(scores)
    nonzero_idx = np.where(scores > 0)[0]
    if len(nonzero_idx) == 0:
        return np.zeros(len(scores), dtype=int)
    k = min(k, len(nonzero_idx))
    top_k = nonzero_idx[np.argsort(scores[nonzero_idx])[-k:]]
    binary = np.zeros(len(scores), dtype=int)
    binary[top_k] = 1
    return binary

def mask_words(inputs, tokenizer, keep_word_mask, text):
    input_ids = inputs["input_ids"].clone()  
    attention_mask = inputs["attention_mask"].clone()    
    word_ids = inputs.get("word_ids")
    if word_ids is None:
        tokenized = tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        word_ids = tokenized.word_ids()   
    mask_token_id = tokenizer.mask_token_id or tokenizer.unk_token_id    
    current_word  = -1    
    for pos in range(1, input_ids.size(1)):  
        if word_ids[pos] is None:continue
        if word_ids[pos] != current_word:
            current_word += 1
            if current_word >= len(keep_word_mask): break
            keep = keep_word_mask[current_word] 
        if not keep:input_ids[0, pos] = mask_token_id
    return {"input_ids": input_ids, "attention_mask": attention_mask}
    
def get_class_probability(model, inputs, class_idx, output_attentions=False):
    with torch.no_grad():
        model_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
        outputs = model(**model_inputs, output_attentions=output_attentions)
        logits = outputs.logits 
        probs = torch.softmax(logits, dim=-1)
        prob = probs[0, class_idx].clamp(0.0, 1.0).item()
    return prob


def get_lime_importance(model, text, tokenizer, target_class, max_features=30, num_samples=1000):
    """LIME word-level importance."""
    def predict_proba(texts):
        enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
        return probs

    explainer = LimeTextExplainer(
        class_names=['NON 0', 'CYB 1', 'HAT 2', 'PRO 3', 'OTH 4'],
        split_expression=r"\s+",
        bow=False,
        kernel_width=25,
        random_state=42,
    )
    exp = explainer.explain_instance(
        text_instance=text,
        classifier_fn=predict_proba,
        num_features=max_features,
        num_samples=num_samples,
        labels=[target_class],
    )
    lime_word_scores = dict(exp.as_list(label=target_class))
    words = text.split()
    word_importance = np.zeros(len(words), dtype=np.float32)
    for i, w in enumerate(words):
        if w in lime_word_scores:
            word_importance[i] = lime_word_scores[w]
    return word_importance


In [6]:

results_attn = { "auprc": [],"token_f1": [], "iou": [],"comprehensiveness": [], "sufficiency": []}
attn_miss = 0
REMOVE_CHARS = r"[|۔؟\?,\.،!_:\"\-_]"
for _, row in test_df[test_df[ycolumn] != 0].iterrows():  
    text = row[xcolumn]
    text = re.sub(REMOVE_CHARS, "", text)
    gold_word_rationale = np.array(row["rationale"])
    if len(gold_word_rationale) == 0: continue
    gold_label = int(row[ycolumn])  
    tokenized = tokenizer(text,truncation=True,
        padding="max_length",max_length=MAX_LENGTH,
        return_offsets_mapping=True,return_tensors="pt",)
    word_ids = tokenized.word_ids()
    inputs = {"input_ids": tokenized["input_ids"].to(device),
        "attention_mask": tokenized["attention_mask"].to(device),
        "word_ids": word_ids,}
    orig_prob = get_class_probability(model, inputs, gold_label, output_attentions=False)  
    with torch.no_grad(): 
        outputs = model(**inputs, output_attentions=True)
    attentions = outputs.attentions
    last_layer_attn = attentions[-1].squeeze(0)
    avg_last_attn = last_layer_attn.mean(dim=0)
    token_importance = avg_last_attn[0].cpu().numpy()
    token_importance = np.maximum(token_importance, 0.0)
    token_importance[0] = 0.0
    word_importance = subword_to_word_importance(token_importance, word_ids)

    
    # align lengths (same as original)
    while len(gold_word_rationale) > len(word_importance):
        zeros = np.where(gold_word_rationale == 0)[0]
        idx = zeros[0] if len(zeros) > 0 else np.where(gold_word_rationale == 1)[0][0]
        gold_word_rationale = np.delete(gold_word_rationale, idx)
    while len(gold_word_rationale) < len(word_importance):
        gold_word_rationale = np.append(gold_word_rationale, 0)

    if len(word_importance) != len(gold_word_rationale):
        print("Length mismatch — skipping")
        attn_miss += 1
        continue

    k = min(8, len(word_importance))
    pred_word_binary = binarize_topk_nonzero(word_importance, k=k)

    results_attn["auprc"].append(auprc(word_importance, gold_word_rationale))
    results_attn["token_f1"].append(token_f1(pred_word_binary, gold_word_rationale))
    results_attn["iou"].append(iou(pred_word_binary, gold_word_rationale))

    # Faithfulness (now using gold class probability)
    comp_keep_mask = pred_word_binary == 0
    masked_comp_inputs = mask_words(inputs, tokenizer, comp_keep_mask, text)
    comp_prob = get_class_probability(model, masked_comp_inputs, gold_label, output_attentions=False)
    results_attn["comprehensiveness"].append(orig_prob - comp_prob)

    suff_keep_mask = pred_word_binary == 1
    masked_suff_inputs = mask_words(inputs, tokenizer, suff_keep_mask, text)
    suff_prob = get_class_probability(model, masked_suff_inputs, gold_label, output_attentions=False)
    results_attn["sufficiency"].append(orig_prob - suff_prob)

NameError: name 'get_class_probability' is not defined

In [11]:
print("\nDataset-level results\n")
print("attn_miss:",attn_miss)
for name, values in results_attn.items():
    if not values:
        print(f"{name:18}: no valid examples")
        continue
    mean = np.mean(values)
    std  = np.std(values)
    n    = len(values)
    print(f"{name:18}: {mean:.4f} ± {std:.4f}  (n={n})")

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
with open(result_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["--- ATTENTION RESULTS ---", timestamp])
    for metric_name, values in results_attn.items():
        mean_val = np.mean(values)
        std_val  = np.std(values)
        writer.writerow([metric_name,"-ATTN--",r2(mean_val),r2(std_val)])
print("Result Saved")


Dataset-level results

attn_miss: 0
auprc             : 0.6097 ± 0.2728  (n=1732)
token_f1          : 0.4423 ± 0.2144  (n=1732)
iou               : 0.3103 ± 0.1950  (n=1732)
comprehensiveness : 0.5198 ± 0.4137  (n=1732)
sufficiency       : 0.1127 ± 0.3363  (n=1732)
Result Saved


In [8]:
results_lime = { "auprc": [],"token_f1": [], "iou": [],"comprehensiveness": [], "sufficiency": []}
start_idx = 0
try:
    existing_df = pd.read_csv(SAVE_result)
    if not existing_df.empty:
        for col in results_lime.keys():
            if col in existing_df.columns:
                results_lime[col] = existing_df[col].tolist()       
        start_idx = len(results_lime["auprc"])  # number already processed
        print(f"Resuming from row {start_idx} (already processed {start_idx} examples)")
        print(f"Current datetime: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    else:
        print("CSV exists but is empty → starting from scratch")
except FileNotFoundError:
    print("No existing CSV found → starting from scratch")
except Exception as e:
    print(f"Error loading CSV: {e} → starting from scratch")
i = start_idx 
positive_rows = test_df[test_df[ycolumn] != 0].reset_index(drop=True)
remaining_rows = positive_rows.iloc[start_idx:]
print(f"Total positive rows: {len(positive_rows)}")
print(f"Already processed: {start_idx}")
print(f"Remaining to process: {len(remaining_rows)}")

No existing CSV found → starting from scratch
Total positive rows: 1732
Already processed: 0
Remaining to process: 1732


In [9]:
for idx, row in remaining_rows.iterrows():
    if i % 100 == 0:
        print(f"Processed {i} / {len(positive_rows)}, saving full results... "
              f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        pd.DataFrame(results_lime).to_csv(SAVE_result, index=False)
    i += 1
    text = row[xcolumn]
    gold_word_rationale = np.array(row["rationale"])
    if len(gold_word_rationale) == 0:continue
    gold_label = int(row[ycolumn])
    tokenized = tokenizer(text,truncation=True,
        padding="max_length",max_length=MAX_LENGTH,
        return_offsets_mapping=True,return_tensors="pt",)
    word_ids = tokenized.word_ids()
    inputs = {"input_ids": tokenized["input_ids"].to(device),
        "attention_mask": tokenized["attention_mask"].to(device),
        "word_ids": word_ids,}
    lime_scores = get_lime_importance(model=model,
        text=text,tokenizer=tokenizer,target_class=gold_label, 
        max_features=len(gold_word_rationale),num_samples=2000,)

    word_importance = np.maximum(lime_scores, 0.0)
    word_importance = word_importance / (word_importance.sum() + 1e-12)
    orig_prob = get_class_probability(model, inputs, gold_label)
    if len(word_importance) != len(gold_word_rationale):
        print(f"Length mismatch (example {idx}) — skipping")
        continue
    k = min(8, len(word_importance))
    pred_word_binary = binarize_topk_nonzero(word_importance, k=k)
    results_lime["auprc"].append(auprc(word_importance, gold_word_rationale))
    results_lime["token_f1"].append(token_f1(pred_word_binary, gold_word_rationale))
    results_lime["iou"].append(iou(pred_word_binary, gold_word_rationale))
    comp_keep_mask = pred_word_binary == 0
    masked_comp_inputs = mask_words(inputs, tokenizer, comp_keep_mask, text)
    comp_prob = get_class_probability(model, masked_comp_inputs, gold_label, output_attentions=False)
    results_lime["comprehensiveness"].append(orig_prob - comp_prob)
    suff_keep_mask = pred_word_binary == 1
    masked_suff_inputs = mask_words(inputs, tokenizer, suff_keep_mask, text)
    suff_prob = get_class_probability(model, masked_suff_inputs, gold_label, output_attentions=False)
    results_lime["sufficiency"].append(orig_prob - suff_prob)
# Final save after loop completes
pd.DataFrame(results_lime).to_csv(SAVE_result, index=False)
print(f"Finished! Total processed: {len(results_lime['auprc'])}")
print(f"Results saved to: {SAVE_result}")
print(f"Final datetime: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Processed 0 / 1732, saving full results... 2026-03-08 05:06:53
Processed 100 / 1732, saving full results... 2026-03-08 05:12:31
Processed 200 / 1732, saving full results... 2026-03-08 05:18:02
Processed 300 / 1732, saving full results... 2026-03-08 05:23:27
Processed 400 / 1732, saving full results... 2026-03-08 05:28:53
Processed 500 / 1732, saving full results... 2026-03-08 05:34:27
Processed 600 / 1732, saving full results... 2026-03-08 05:40:08
Processed 700 / 1732, saving full results... 2026-03-08 05:45:32
Processed 800 / 1732, saving full results... 2026-03-08 05:51:18
Processed 900 / 1732, saving full results... 2026-03-08 05:57:00
Processed 1000 / 1732, saving full results... 2026-03-08 06:02:46
Processed 1100 / 1732, saving full results... 2026-03-08 06:08:36
Processed 1200 / 1732, saving full results... 2026-03-08 06:14:32
Processed 1300 / 1732, saving full results... 2026-03-08 06:20:19
Processed 1400 / 1732, saving full results... 2026-03-08 06:26:02
Processed 1500 / 1732,

In [10]:
print("\nLIME-based Dataset-level results:\n")
for name, values in results_lime.items():
    if not values:
        print(f"{name:18}: no valid examples")
        continue
    mean = np.mean(values)
    std  = np.std(values)
    n    = len(values)
    print(f"{name:18}: {mean:.4f} ± {std:.4f}  (n={n})")

with open(result_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["--- LIME RESULTS ---"])
    for metric_name, values in results_lime.items():
        mean_val = np.mean(values)
        std_val  = np.std(values)
        writer.writerow([metric_name,"-LIME--",r2(mean_val),r2(std_val)])
print("File Saved")


LIME-based Dataset-level results:

auprc             : 0.6205 ± 0.2634  (n=1732)
token_f1          : 0.4173 ± 0.2061  (n=1732)
iou               : 0.2871 ± 0.1831  (n=1732)
comprehensiveness : 0.5788 ± 0.3963  (n=1732)
sufficiency       : -0.1102 ± 0.3205  (n=1732)
File Saved


In [ ]:
script_name = os.path.basename(__file__) if "__file__" in globals() else "interactive"
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("######################################################################")
print(f"Model: {modelname} | Script: {script_name} | Timestamp: {timestamp}\n")
print('Accuracy:',r2(report["accuracy"]))
print(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
for class_name, metrics in report.items():
    if isinstance(metrics, dict):
        print(f"{class_name:<15} {r2(metrics['precision']):<10} {r2(metrics['recall']):<10} "
              f"{r2(metrics['f1-score']):<10} {r2(metrics['support']):<10}")
# Token-level / Faithfulness metrics
print("\nToken-level / Faithfulness Metrics (Mean ± Std):")
print(f"{'Metric':<20} {'Attention':<15} {'LIME':<15}")
for metric_name in results_attn.keys():
    attn_mean = r2(np.mean(results_attn[metric_name]))
    attn_std = r2(np.std(results_attn[metric_name]))
    lime_mean = r2(np.mean(results_lime[metric_name]))
    lime_std = r2(np.std(results_lime[metric_name]))
    print(f"{metric_name:<20} {attn_mean} ± {attn_std:<11} {lime_mean} ± {lime_std:<11}")
